In [1]:
# Code bellow is partiala based on photobook code

In [1]:
import os
import re
import datetime
from collections import defaultdict
import json
import re
import datetime
from collections import defaultdict
from matplotlib import pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import base64
import requests

In [2]:
class Log:
    def __init__(self, logfile):
        self.game_id = logfile['game_id']
        self.domain_id = logfile['domain_id']
        self.agent_ids = logfile['agent_ids']
        self.agent_labels = logfile['agent_labels']
        self.feedback = logfile['feedback']
        self.rounds, self.complete = self.load_rounds(logfile['rounds'], self.game_id)
        self.total_score = self.calculate_score()
        self.scores = self.calculate_player_scores()
        self.start_time = logfile['start_time']
        self.duration = self.calculate_duration()
        self.domains = self.get_domains()
        self.check_feedback()

    def load_rounds(self, game_rounds, game_id):
        rounds = []
        message_id = 0
        for round_data in game_rounds:
            game_round = GameRound(round_data, game_id, message_id)
            message_id = game_round.message_id
            rounds.append(game_round)
        if len(rounds) < 5:
            return (rounds, False)
        return (rounds, True)

    def calculate_score(self):
        total_score = 0
        for game_round in self.rounds:
            total_score += game_round.total_score
        return total_score

    def calculate_player_scores(self):
        player_scores = defaultdict(lambda: 0)
        for game_round in self.rounds:
            for player, score in game_round.scores.items():
                player_scores[player] += score
        return player_scores

    def get_domains(self):
        path = self.rounds[0].images["A"][0].split("/")[0]
        return [domain for domain in path.rsplit("_", 1)]

    def calculate_duration(self):
        start_time = self.rounds[0].messages[0].timestamp
        end_time = self.rounds[-1].messages[-1].timestamp
        return end_time - start_time

    def check_feedback(self):
        if "A" not in self.feedback:
            self.feedback["A"] = None
        if "B" not in self.feedback:
            self.feedback["B"] = None

    def format_time(datetime_obj):
        return datetime_obj.strftime('%M:%S')


    def strip_image_id(image_path):
        return int(image_path.split('_')[-1].split('.')[0].lstrip('0'))


class GameRound:
    def __init__(self, logfile_entry, game_id, message_id):
        self.round_nr = logfile_entry['round_nr'] + 1
        self.images = logfile_entry['images']
        self.common = logfile_entry['common']
        self.highlighted = logfile_entry['highlighted']
        self.scores = dict(logfile_entry['score'])
        self.total_score = self.calculate_score(logfile_entry['score'])
        self.messages, self.message_id = self.load_messages(logfile_entry['messages'], message_id)
        self.num_messages = self.count_text_messages()
        self.duration = self.calculate_duration()

    def load_messages(self, message_list, message_id):
        messages = []
        for message_data in message_list:
            message = Message(message_data, message_id)
            messages.append(message)
            message_id += 1

        if len(messages) == 0:
            print('ERROR: Missing messages for this game')
        return messages, message_id

    def calculate_score(self, score_dict):
        score = 0
        if not score_dict.values():
            return None
        for player_score in score_dict.values():
            score += player_score
        return score

    def count_text_messages(self):
        count = 0
        for message in self.messages:
            if message.type == "text":
                count += 1
        return count

    def calculate_duration(self):
        start_time = self.messages[0].timestamp
        for message in self.messages[::-1]:
            if message.type == 'feedback':
                end_time = message.timestamp
                return end_time - start_time


class Message:
    def __init__(self, logfile_message, message_id):
        self.message_id = message_id
        self.agent_id = logfile_message['agent_id']
        self.text = logfile_message['message']
        self.speaker = logfile_message['speaker']
        if message_id == 0:
            self.timestamp = datetime.datetime.strptime(logfile_message['timestamp'], '%H:%M:%S')
        else:
            self.timestamp = datetime.datetime.strptime(logfile_message['timestamp'], '%H:%M:%S.%f')
        self.turn = logfile_message['turn']
        self.type = self.determine_message_type()

    def determine_message_type(self):
        if not self.text.startswith("<"):
            return "text"
        else:
            return re.findall(r'<(.*?)>', self.text)[0]

In [3]:
def load_logs(log_repository, data_path):

    filepath = os.path.join(data_path, log_repository)
    print("Loading logs from {}...".format(filepath))

    missing_counter = 0
    file_count = 0
    for _, _, files in os.walk(filepath):
        file_count += len(files)
    print("{} files found.".format(file_count))
    logs = []
    for root, dirs, files in os.walk(filepath):
        for file in files:
            if file.endswith(".json"):
                with open(os.path.join(root, file), 'r') as logfile:
                    log = Log(json.load(logfile))
                    if log.complete:
                        logs.append(log)

    print("DONE. Loaded {} completed game logs.".format(len(logs)))
    return logs


In [4]:
data_path = ""
logs = load_logs("logs", data_path)

Loading logs from logs...
2504 files found.
DONE. Loaded 2504 completed game logs.


In [5]:
def print_transcript(log):
    print("Game ID: {}".format(log.game_id))
    print("Domain ID: {}".format(log.domain_id))
    print("Image set main objects: '{}' and '{}'".format(log.domains[0], log.domains[1]))
    print("Participant IDs: {} and {}".format(log.agent_ids[0], log.agent_ids[1]))
    print("Start Time: {}".format(log.start_time))
    print("Duration: {}".format(log.duration))
    print("Total Score: {}".format(log.total_score))
    print("Player scores: A - {}, B - {}".format(log.scores["A"], log.scores["B"]))
    print("Transcript:\n")

    for round_data in log.rounds:
        print("Round {}".format(round_data.round_nr))
        for message in round_data.messages:
            if message.type == "text":
                print("[{}] {}: {}".format(Log.format_time(message.timestamp), message.speaker, message.text))

            if message.type == "selection":
                label = "common" if message.text.split()[1] == "<com>" else "different"
                print("[{}] {} marks image {} as {}".format(Log.format_time(message.timestamp), message.speaker, Log.strip_image_id(message.text.split()[2]), label))

        print("\nDuration: {}".format(round_data.duration))
        print("Total Score: {}".format(round_data.total_score))        
        print("Player scores: A - {}, B - {}".format(round_data.scores["A"], round_data.scores["B"]))
        print("Number of messages: {}\n".format(round_data.num_messages))


In [15]:
print_transcript(logs[1])

Game ID: 2106
Domain ID: 55
Image set main objects: 'dining_table' and 'refrigerator'
Participant IDs: 4 and 5
Start Time: 2018-05-17T13:54:42.825720
Duration: 0:10:49.054509
Total Score: 25
Player scores: A - 12, B - 13
Transcript:

Round 2
[00:00] A: I have a picture of a plant on a table with a yellow fridge on the left. Three rows of shelves.
[00:04] B: guy in kitchen being redone. big white fridge
[00:15] A: I do not have that.
[00:27] B: no i dont have the plant
[00:31] B marks image 177775 as different
[00:46] A: I have a woman with yellow gloves spraying a table with one hand and sponging with other.
[00:49] A marks image 43256 as different
[00:54] B: long wood table with yellow cabinet and supplies
[01:01] B: i have the woman
[01:03] B marks image 130011 as common
[01:07] A marks image 130011 as common
[01:25] A: Yellow cabinet is on left? I thought that was a fridge. I have that if there is a plant on the tabl
[01:38] B: yes thats it
[01:41] A: I have young boy eating hotdog 

In [16]:
set_labels = set()
for log in logs:
    for round_data in log.rounds:
        for message in round_data.messages:
            if message.type == "selection":
                set_labels.add(message.text.split()[1])
print(set_labels)

{'<dif>', '<com>'}


In [11]:
def create_round_dict(log_list):
    r_list = []
    for log in log_list:
        for round_data in log.rounds:
            # only rounds that are 100% correct
            if round_data.total_score == 6:
                r_list.append((round_data, log.game_id))
    return r_list

In [12]:
round_list_tpl = create_round_dict(logs)

In [13]:
round_list_tpl[7][0].total_score

6

In [10]:
len(round_list_tpl)

9626

In [11]:
# # # Create tables of images
# # def create_representation_sets(sets_list):
# #     for set_tpl in sets_list:
# #             # base_list = round_list_tpl[0][0].images['A']
# #         game_number = set_tpl[1]
# #         set_number = set_tpl[0].round_nr
# #         for player in set_tpl[0].images:
# #             base_list = set_tpl[0].images[player]
# #             img_list = [Image.open("images/" + f) for f in base_list]
# #             w, h = img_list[0].size
# #             w //= 3
# #             h //= 3
# #             img_list = [im.resize((w, h)) for im in img_list]
# #             cols, rows = 3, 2
# #             table = Image.new("RGB", (cols * w, rows * h))
            
# #             for idx, im in enumerate(img_list):
# #                 x = (idx % cols) * w
# #                 y = (idx // cols) * h
# #                 table.paste(im, (x, y))
# #             table.save("sets_images/{}_{}_{}.jpg".format(game_number, set_number, player))

# # create_representation_sets(round_list_tpl)

In [14]:
def create_representation_sets_strip(sets_list, strip=10):
    counter = 20
    for set_tpl in sets_list:
        if counter < 15:
            game_number = set_tpl[1]
            set_number = set_tpl[0].round_nr
            for player in set_tpl[0].images:
                base_list = set_tpl[0].images[player]
                img_list = [Image.open("images/" + f) for f in base_list]
    
                w, h = img_list[0].size
                w //= 3
                h //= 3
                img_list = [im.resize((w, h)) for im in img_list]
    
                cols, rows = 3, 2
    
                total_w = cols * w + (cols - 1) * strip
                total_h = rows * h + (rows - 1) * strip
                table = Image.new("RGB", (total_w, total_h), "black")
    
                for idx, im in enumerate(img_list):
                    col = idx % cols
                    row = idx // cols
                    x = col * (w + strip)
                    y = row * (h + strip)
                    table.paste(im, (x, y))
    
                table.save(f"sets_images_stripe/{game_number}_{set_number}_{player}.jpg")
                counter += 1
create_representation_sets_strip(round_list_tpl)

In [12]:
def display_round_llava(round_tuple):
    round_data = round_tuple[0]
    game_id = round_tuple[1]
    round_name = "{}_{}".format(game_id, round_data.round_nr)
    
    image_A = Image.open(f"sets_images/{round_name}_A.jpg")
    image_B = Image.open(f"sets_images/{round_name}_B.jpg")

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(image_A)
    axes[0].set_title("A's view")
    axes[0].axis('off')
    axes[1].imshow(image_B)
    axes[1].set_title("B's view")
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()


    print("Round {}".format(round_name))
    print("\n")

    print("LLAVA Description of A")
    # ! curl http://localhost:11434/api/generate -d '{"model": "llava","prompt": "Describe this image.","stream": false,"images": ["'"$(base64 -w 0 sets_images/774_5_A.jpg)"'"]}' | jq -r '.response'
    ! curl http://localhost:11434/api/generate \
          -d '{{"model":"llava","prompt":"Describe this image.","stream":false,"images":["'"$(base64 -w 0 sets_images/{round_name}_A.jpg)"'"]}}' \
          | jq -r '.response'
    print("\n")
    
    print("LLAVA Description of B")
    ! curl http://localhost:11434/api/generate \
          -d '{{"model":"llava","prompt":"Describe this image.","stream":false,"images":["'"$(base64 -w 0 sets_images/{round_name}_B.jpg)"'"]}}' \
          | jq -r '.response'

    print("\n")
    print("Dialogue from dataset")

    for message in round_data.messages:
        if message.type == "text":
            print("{}: {}".format(message.speaker, message.text))

        if message.type == "selection":
            label = "common" if message.text.split()[1] == "" else "different"
            print("{} marks image {} as {}".format(message.speaker, Log.strip_image_id(message.text.split()[2]), label))


In [14]:
display_round_llava(round_list_tpl[0])

FileNotFoundError: [Errno 2] No such file or directory: '/home/gusloryst@GU.GU.SE/mgr/image_processing/PhotoBook/sets_images/1048_2_A.jpg'

In [18]:
img_id = "774_5"

img_path = f"sets_images/{img_id}_A.jpg"

with open(img_path, "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode("utf-8")

payload = {
    "model": "llava:34b",
    "prompt": "Describe this image.",
    "stream": False,
    "images": [img_b64],
}

resp = requests.post(
    "http://localhost:11434/api/generate",
    data=json.dumps(payload),
    headers={"Content-Type": "application/json"},
)

resp.raise_for_status()
print(resp.json()["response"])


The image shows a collage of six separate photographs, each depicting an individual or two individuals sitting on park benches. In the top left photo, there is one person; in the top right, there are two people, one wearing robes suggesting they could be a monk; the bottom left features another single person; and the bottom center also shows two individuals. The people appear to be relaxed, some looking at the ground, and there's no visible text on their clothing or in the environment.

The style of the image is that of a traditional collage with a white border separating each photo, providing an overview of park benches as places for rest and contemplation. The individuals are dressed casually, appropriate for outdoor activities, suggesting a calm, leisurely atmosphere.


In [30]:
def display_round_stripe(round_tuple, show=False):

    if show == True:
        round_data = round_tuple[0]
        game_id = round_tuple[1]
        round_name = "{}_{}".format(game_id, round_data.round_nr)
        
        image_A = Image.open(f"sets_images_stripe/{round_name}_A.jpg")
        image_B = Image.open(f"sets_images_stripe/{round_name}_B.jpg")
    
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
        axes[0].imshow(image_A)
        axes[0].set_title("A's view")
        axes[0].axis('off')
        axes[1].imshow(image_B)
        axes[1].set_title("B's view")
        axes[1].axis('off')
    
        plt.tight_layout()
        plt.show()
    
    
        print("Round {}".format(round_name))
        print("\n")
    
        for message in round_data.messages:
            if message.type == "text":
                print("{}: {}".format(message.speaker, message.text))
    
            if message.type == "selection":
                label = "common" if message.text.split()[1] == "" else "different"
                print("{} marks image {} as {}".format(message.speaker, Log.strip_image_id(message.text.split()[2]), label))

    print(round_tuple[0].images)


In [31]:
display_round_stripe(round_list_tpl[7], show=False)

{'A': ['chair_couch/COCO_train2014_000000131927.jpg', 'chair_couch/COCO_train2014_000000461340.jpg', 'chair_couch/COCO_train2014_000000002347.jpg', 'chair_couch/COCO_train2014_000000512515.jpg', 'chair_couch/COCO_train2014_000000399122.jpg', 'chair_couch/COCO_train2014_000000408925.jpg'], 'B': ['chair_couch/COCO_train2014_000000581136.jpg', 'chair_couch/COCO_train2014_000000512515.jpg', 'chair_couch/COCO_train2014_000000399122.jpg', 'chair_couch/COCO_train2014_000000119002.jpg', 'chair_couch/COCO_train2014_000000461340.jpg', 'chair_couch/COCO_train2014_000000408925.jpg']}
